#different visualization in phytclust

In [ ]:
from phytclust.main import PhytClust
from phytclust import plot_tree, plot_multiple_clusters
from phytclust import pairwise_distances
import io
import os
from Bio import Phylo
import sys
import numpy as np
import random
import matplotlib as mpl
import os
import datetime
import matplotlib.pyplot as plt
import matplotlib as mpl
import logging
from collections import defaultdict
from typing import Any, Callable, Dict, List, Optional, Tuple, Union, Optional
mpl.rc("font", family="monospace")  # or "Helvetica", "Times New Roman", etc.

import pandas as pd

1. tree plotted with scale bar and leaf nodes coloured by cluster, and opitonal box around them or not. graph can be sraight or circular, custom color map

In [ ]:
from matplotlib.patches import Rectangle
COLORS = {
    "allele_a": mpl.colors.to_rgba("orange"),
    "allele_b": mpl.colors.to_rgba("teal"),
    "clonal": mpl.colors.to_rgba("lightgrey"),
    "normal": mpl.colors.to_rgba("dimgray"),
    "gain": mpl.colors.to_rgba("red"),
    "wgd": mpl.colors.to_rgba("green"),
    "loss": mpl.colors.to_rgba("blue"),
    "chr_label": mpl.colors.to_rgba("grey"),
    "vlines": "#1f77b4",
    "marker_internal": "#1f77b4",
    "marker_terminal": "black",
    "marker_normal": "green",
    "summary_label": "grey",
    "background": "white",
    "background_hatch": "lightgray",
    "patch_background": "white",
}
LINEWIDTHS = {
    "copy_numbers": 2,
    "chr_boundary": 1,
    "segment_boundary": 0.5,
}
ALPHAS = {
    "patches": 0.15,
    "patches_wgd": 0.3,
    "clonal": 0.3,
}
SIZES = {
    "tree_marker": 40,
    "ylabel_font": 8,
    "ylabel_tick": 6,
    "xlabel_font": 10,
    "xlabel_tick": 8,
    "chr_label": 8,
}


def _get_x_positions(tree: Any) -> Dict[Any, float]:
    """Create a mapping of each clade to its horizontal position.
    Dict of {clade: x-coord}
    """
    depths = tree.depths()
    # If there are no branch lengths, assume unit branch lengths
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths


# def _get_y_positions(
#     tree: Any, adjust: bool = False, outgroup: str = "outgroup"
# ) -> Dict[Any, float]:
#     """Create a mapping of each clade to its vertical position.
#         Dict of {clade: y-coord}.
#     Coordinates are negative, and integers for tips.
#     """
#     maxheight = tree.count_terminals()
#     heights = {
#         tip: maxheight - 1 - i
#         for i, tip in enumerate(
#             reversed(
#                 [
#                     x
#                     for x in tree.get_terminals()
#                     if outgroup is None or x.name != outgroup
#                 ]
#             )
#         )
#     }
#     if outgroup is not None:
#         normal_clades = list(tree.find_clades(outgroup))
#         if not normal_clades:
#             raise PlotError(f"Normal clade {outgroup} not found in tree")
#         heights[normal_clades[0]] = maxheight

#     # Internal nodes: place at midpoint of children
#     def calc_row(clade: Any) -> None:
#         for subclade in clade:
#             if subclade not in heights:
#                 calc_row(subclade)
#         # Closure over heights
#         heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

#     if tree.root.clades:
#         calc_row(tree.root)

#     if adjust:
#         pos = pd.DataFrame(
#             [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
#         ).sort_values("pos")
#         pos["newpos"] = 0
#         count = 0
#         for i in pos.index:
#             if pos.loc[i, "clade"] != tree.root:
#                 count += 1
#             pos.loc[i, "newpos"] = count

#         pos.set_index("clade", inplace=True)
#         heights = pos.to_dict()["newpos"]

#     return heights


def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: str = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
        Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    leaves = [x for x in tree.get_terminals() if outgroup is None or x.name != outgroup]
    heights = {tip: maxheight - 1 - i for i, tip in enumerate(reversed(leaves))}

    # heights = {
    #     tip: maxheight - 1 - i
    #     for i, tip in enumerate(
    #         reversed(
    #             [
    #                 x
    #                 for x in tree.get_terminals()
    #                 if outgroup is None or x.name != outgroup
    #             ]
    #         )
    #     )
    # }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade {outgroup} not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        if clade.clades:
            # The clade's position is the midpoint of its first and last child's position
            first_child = clade.clades[0]
            last_child = clade.clades[-1]
            heights[clade] = (heights[first_child] + heights[last_child]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        sorted_heights = sorted(heights.items(), key=lambda x: x[1])
        new_heights = {}
        count = 0
        for clade, _ in sorted_heights:
            # Keep the root at the initial reference, start counting from there
            if clade != tree.root:
                count += 1
            new_heights[clade] = count
        heights = new_heights

    return heights


def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = True,
    hide_branch_length_axis: bool = True,
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    clade_color_map: Optional[Dict[Any, str]] = None,
    cluster_mrcas: Optional[
        List[Any]
    ] = None,  # New parameter: List of MRCA clades for clusters
    cluster_labels: Optional[
        List[str]
    ] = None,  # New parameter: Labels for these clusters
    **kwargs: Any,
) -> plt.Figure:
    """
    Plot the given tree using matplotlib, and optionally highlight cluster MRCAs.
    cluster_mrcas: list of clade objects representing MRCAs of clusters you want to highlight
    cluster_labels: list of strings for labeling these clusters. Should match length of cluster_mrcas.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        try:
            import pylab as plt
        except ImportError:
            raise MEDICCPlotError(
                "Install matplotlib or pylab if you want to use draw."
            ) from None

    import matplotlib.collections as mpcollections

    marker_size = marker_size or SIZES["tree_marker"]
    line_width = line_width or LINEWIDTHS.get("segment_boundary", 1.0)
    label_func = label_func or (lambda x: x.name if hasattr(x, "name") else x)

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = np.max(
            [
                np.sum([x.branch_length for x in input_tree.get_path(leaf)])
                for leaf in input_tree.get_terminals()
            ]
        )
        plot_width = 5 + np.max([0, width_scale * 0.5])
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))

    get_label_color = lambda label: "black"
    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = (
                COLORS["marker_terminal"] if is_terminal else COLORS["marker_internal"]
            )
            if sample == outgroup:
                clade_colors[sample] = COLORS["marker_normal"]
        get_label_color = lambda label: clade_colors.get(label, "black")

    marker_size = marker_size if marker_size is not None else SIZES["tree_marker"]
    marker_func = lambda x: (
        (marker_size, get_label_color(x.name)) if x.name is not None else None
    )

    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=SIZES["xlabel_tick"])
    ax.xaxis.label.set_size(SIZES["xlabel_font"])
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )
    x_posns = _get_x_positions(input_tree)
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )

    # optionally ax.spines["bottom"].set_visible(False)
    xmax = max(x_posns.values())
    ax.set_xlim(-0.05 * xmax, 1.05 * xmax)
    top_margin = 0.5
    ymax = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if not branch_labels:
        if show_branch_lengths:

            def format_branch_label(x):
                return (
                    value_to_str(np.round(x.branch_length, 1))
                    if x.name != "root" and x.name is not None
                    else None
                )

        else:

            def format_branch_label(clade):
                return None

    elif isinstance(branch_labels, dict):

        def format_branch_label(clade):
            return branch_labels.get(clade)

    else:
        if not callable(branch_labels):
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

        def format_branch_label(clade):
            return value_to_str(branch_labels(clade))

    if show_branch_support:

        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, default_color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]

        line_color = (
            clade_color_map.get(clade, default_color)
            if clade_color_map
            else default_color
        )

        if hasattr(clade, "color") and clade.color is not None:
            line_color = clade.color.to_hex()
        if hasattr(clade, "width") and clade.width is not None:
            lw = clade.width * plt.rcParams["lines.linewidth"]

        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=line_color,
            lw=lw,
        )

        if marker_func is not None:
            marker = marker_func(clade)
            if (
                marker is not None
                and clade is not None
                and not (hide_internal_nodes and not clade.is_terminal())
            ):
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        label = label_func(str(clade.name))
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top_local = y_posns[clade.clades[0]]
            y_bot_local = y_posns[clade.clades[-1]]

            child_colors = []
            for child in clade.clades:
                child_line_color = (
                    clade_color_map.get(child, line_color)
                    if clade_color_map
                    else line_color
                )
                child_colors.append(child_line_color)

            if len(set(child_colors)) == 1:
                vertical_line_color = child_colors[0]
            else:
                vertical_line_color = line_color

            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot_local,
                y_top=y_top_local,
                color=vertical_line_color,
                lw=lw,
            )

            for child in clade:
                draw_clade(child, x_here, line_color, lw)

    line_width = (
        line_width if line_width is not None else plt.rcParams["lines.linewidth"]
    )
    draw_clade(input_tree.root, 0, "k", line_width)

    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    for x, y, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(x, y, text, verticalalignment="center", color=color)

    if hide_branch_length_axis:
        ax.set_xticks([])
        ax.set_xlabel("")
        ax.spines["bottom"].set_visible(False)
        ax.spines["bottom"].set_visible(False)

    else:
        ax.set_xlabel("branch length")
    ax.set_ylabel("taxa")

    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None

        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    # --- New Code: Highlight cluster MRCA nodes and add a label box ---
    if cluster_mrcas and cluster_labels and len(cluster_mrcas) == len(cluster_labels):
        for mrca, cl_label in zip(cluster_mrcas, cluster_labels):
            # Get MRCA coordinates
            mx = x_posns[mrca]
            my = y_posns[mrca]

            # Draw a semi-transparent marker at the MRCA node
            # E.g., a circle marker with alpha=0.5 in the cluster's color
            node_color = clade_color_map.get(mrca, "red")
            ax.scatter(
                mx, my, s=100, c=node_color, alpha=0.5, zorder=5, edgecolors="none"
            )

            # Create a small box for the label to the right of the node
            box_width = 1.0  # Arbitrary width
            box_height = 0.5  # Arbitrary height
            box_x = mx + 0.1  # small offset from the node
            box_y = my - box_height / 2  # center vertically around the node

            # Add rectangle patch
            rect = Rectangle(
                (box_x, box_y),
                box_width,
                box_height,
                facecolor=node_color,
                alpha=0.5,
                zorder=4,
            )
            ax.add_patch(rect)

            # Add text inside the box
            ax.text(
                box_x + box_width / 2,
                box_y + box_height / 2,
                cl_label,
                ha="center",
                va="center",
                color="black",
                fontsize=8,
            )

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()

from Bio import Phylo
import matplotlib.pyplot as plt
import numpy as np
from Bio import Phylo
import matplotlib.pyplot as plt

tree_string = "(((A:5, B:3)C1:6, (C:3, D:7)D1:4)A13:22, (((E:7, F:13)E12:5, G:6)B23:10, H:60)EF:35)GH:0;"
tree = Phylo.read(io.StringIO(tree_string), "newick")
# Define your clusters
clusters = [
    ["A", "B"],  # Cluster 1
    ["D", "C"],  # Cluster 2
    ["E", "F", "G", "H"],  # Cluster 3
]

# Assign colors for each cluster
cluster_colors = ["red", "blue", "green"]

# Create a map from clade to color
clade_color_map = {}
for i, cluster in enumerate(clusters):
    color = cluster_colors[i]
    mrca = tree.common_ancestor(cluster)

    # Color the entire subtree under this MRCA
    for descendant in mrca.find_clades():
        clade_color_map[descendant] = color
    # Remove the MRCA itself so its horizontal line doesn't take the cluster color
    if mrca in clade_color_map:
        del clade_color_map[mrca]

# Ensure you have the plot_tree function code defined above this point
plot_tree(
    tree,
    clade_color_map=clade_color_map,
    line_width=3,
    height_scale=2,
    width_scale=0.5,
    marker_size=0.005,
    hide_internal_nodes=True,
    show_terminal_labels=False,
    label_func=lambda x: "",
)

# Show the plot
plt.show()

In [ ]:
import math
import matplotlib.pyplot as plt
from Bio import Phylo


def compute_radial_positions(tree):
    """
    Return a dict {clade: (r, theta)} for each clade in 'tree',
    where r = distance from root,
          theta = angle in [0..2π] assigned based on tip ordering.
    """
    # 1) gather tips
    tips = tree.get_terminals()
    N = len(tips)
    # sort tips by name or any order
    tips_sorted = sorted(tips, key=lambda x: x.name or "")

    # We'll store radial coords here
    radial_dict = {}

    # 2) assign angles for tips
    for i, tip in enumerate(tips_sorted):
        angle = 2 * math.pi * i / N  # distribute evenly around the circle
        r = tree.distance(tree.root, tip)
        radial_dict[tip] = (r, angle)

    # 3) internal nodes: r = distance from root, angle = average of children
    def assign_angle(clade):
        if clade not in radial_dict:
            child_angles = []
            for child in clade.clades:
                assign_angle(child)
                child_angles.append(radial_dict[child][1])
            # average angle
            angle_mean = sum(child_angles) / len(child_angles)
            r_node = tree.distance(tree.root, clade)
            radial_dict[clade] = (r_node, angle_mean)

    assign_angle(tree.root)
    return radial_dict


def plot_tree_radial(tree, ax=None, line_color="black", line_width=1.0, marker_size=5):
    """
    Plot a radial tree layout using (r,theta) for each node.
    - r: cumulative branch length from root
    - theta: angle assigned among tips, with internal nodes as average
    """
    if ax is None:
        fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(8, 8))
    else:
        fig = ax.figure

    # compute (r,theta) for each node
    radial_pos = compute_radial_positions(tree)

    # store lines (we'll draw them directly)
    lines_r = []
    lines_theta = []
    colors = []

    def draw_clade(clade, parent_r, parent_theta):
        r_here, theta_here = radial_pos[clade]
        # line from parent->child in radial space
        # We'll approximate by 2 points
        # If you want an arc, you can do more interpolation,
        # but let's do a direct radial line for simplicity.
        lines_r.append([parent_r, r_here])
        lines_theta.append([parent_theta, theta_here])
        colors.append(line_color)

        # if the clade has children => recurse
        for child in clade.clades:
            draw_clade(child, r_here, theta_here)

    # root has r=0 or whatever is assigned,
    # but we can start from (0, 0)
    root_r, root_theta = radial_pos[tree.root]
    # ensure root is at r=0 if you want
    radial_pos[tree.root] = (0, root_theta)
    root_r = 0

    for child in tree.root.clades:
        draw_clade(child, root_r, root_theta)

    # Now draw the line segments in polar coords
    # e.g. for each pair (line_r[i], line_theta[i]) do a small plot
    for rvals, tvals, c in zip(lines_r, lines_theta, colors):
        # draw a line in polar coords
        ax.plot(tvals, rvals, color=c, linewidth=line_width)

    # optionally scatter marker at each node
    # radial_pos has each node => (r,theta)
    for clade, (r, theta) in radial_pos.items():
        # is terminal or not?
        is_leaf = not clade.clades
        ax.plot(
            theta,
            r,
            marker="o",
            markersize=(marker_size if is_leaf else marker_size * 0.6),
            color=line_color,
        )

    # set some radial limits
    all_rs = [val[0] for val in radial_pos.values()]
    max_r = max(all_rs)
    ax.set_rmax(max_r * 1.1)  # some margin
    # remove polar labels if you want
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_title("Radial Tree Layout")
    return fig


# USAGE EXAMPLE:
# from Bio import Phylo
# tree = Phylo.read("some_file.newick","newick")
fig = plot_tree_radial(tree)
plt.show()

fig 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from Bio import Phylo
from matplotlib.patches import Polygon, Rectangle
from io import StringIO
from typing import List, Optional

#################################
# 1) UTILS: get_x_positions, get_y_positions, flip_x_positions
#################################

def get_x_positions(tree):
    """
    Return a dict mapping each clade to its x-position (distance from root),
    forcing the root's branch_length=0 to avoid extra stubs.
    """
    # Force root branch length = 0
    if tree.root.branch_length:
        tree.root.branch_length = 0.0

    x_dict = {}
    def assign_x(clade, curx):
        x_dict[clade] = curx
        for child in clade.clades:
            bl = child.branch_length or 0.0
            assign_x(child, curx + bl)
    assign_x(tree.root, 0.0)
    return x_dict


def get_y_positions(tree):
    """
    Return a dict mapping each clade to a y-position in top-to-bottom order for leaves.
    Internal nodes go at the mean of their children.
    """
    y_dict = {}
    y_counter = 0
    def assign_y(clade):
        nonlocal y_counter
        for child in clade.clades:
            assign_y(child)
        if not clade.clades:  # leaf
            y_dict[clade] = y_counter
            y_counter += 1
        else:
            child_ys = [y_dict[ch] for ch in clade.clades]
            y_dict[clade] = np.mean(child_ys)
    assign_y(tree.root)
    return y_dict


def flip_x_positions(x_dict):
    """
    Reflect the x-values around their midpoint so the tree is horizontally reversed.
    """
    xs = list(x_dict.values())
    xmin, xmax = min(xs), max(xs)
    mid = 0.5*(xmin + xmax)
    for c in x_dict:
        old = x_dict[c]
        x_dict[c] = mid - (old - mid)

def scale_x_positions(x_dict, factor: float):
    """
    Multiply all x-coordinates by 'factor' to scale up small branch lengths visually.
    This does not change the 'units' logically, but we must handle the scale bar carefully.
    """
    for c in x_dict:
        x_dict[c] *= factor

#################################
# 2) PLOTTING THE TREE LINES ONLY
#################################

def plot_tree(
    tree,
    ax: plt.Axes,
    flip_x: bool=False,
    scale_factor: float=1.0,
    line_color: str="black",
    line_width: float=1.0
):
    """
    Draws a cladogram (horizontal lines + vertical lines for branches),
    no leaf markers or names. If flip_x=True, data-coords are mirrored horizontally.
    Also multiply all x positions by 'scale_factor'.
    Returns (x_dict, y_dict, scale_factor).
    """
    # 1) get original x,y
    x_dict = get_x_positions(tree)
    y_dict = get_y_positions(tree)

    # 2) optionally flip
    if flip_x:
        flip_x_positions(x_dict)
    # 3) optionally scale up
    if scale_factor != 1.0:
        scale_x_positions(x_dict, scale_factor)

    # gather line segments
    h_segments = []
    v_segments = []

    def add_hline(x1, x2, y):
        h_segments.append(((x1,y),(x2,y)))
    def add_vline(x, y1, y2):
        # skip near-zero lines
        if abs(y2 - y1) < 1e-9:
            return
        v_segments.append(((x,y1),(x,y2)))

    def draw_clade(clade, xstart, is_root=False):
        xh = x_dict[clade]
        yh = y_dict[clade]
        if not is_root:
            add_hline(xstart, xh, yh)
        if clade.clades:
            y_bot = y_dict[clade.clades[-1]]
            y_top = y_dict[clade.clades[0]]
            v_segments.append(((xh,y_bot),(xh,y_top)))
            for child in clade.clades:
                draw_clade(child, xh)

    draw_clade(tree.root, 0.0, is_root=True)

    import matplotlib.collections as mc
    hc = mc.LineCollection(h_segments, colors=line_color, linewidths=line_width)
    vc = mc.LineCollection(v_segments, colors=line_color, linewidths=line_width)
    ax.add_collection(hc)
    ax.add_collection(vc)

    # define lims
    all_x = list(x_dict.values())
    all_y = list(y_dict.values())
    xmin, xmax = min(all_x), max(all_x)
    ymin, ymax = min(all_y), max(all_y)

    xmargin = 0.1*(xmax - xmin) if (xmax>xmin) else 1
    ymargin = 0.5
    ax.set_xlim(xmin - xmargin, xmax + xmargin)
    ax.set_ylim(ymax + ymargin, ymin - ymargin)

    # remove ticks/spines
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    return x_dict, y_dict, scale_factor

#################################
# 3) UNIFORM TRIANGLES + LABELS OUTSIDE
#################################

def add_uniform_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_offset: float=1.0,
    triangle_color: str="red",
    triangle_alpha: float=0.5,
    cluster_labels: Optional[List[str]]=None,
    label_fontsize: int=10
):
    """
    - Compute bounding box from leaves only => uniform chunk in vertical dimension.
    - Triangles each have the same vertical height.
    - Place label boxes *outside* the triangle (to the extreme right or left).

    The top leaf gets the first chunk, the bottom leaf gets the last chunk,
    so they exactly fill the leaf range with optional padding.
    """
    # gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)
    # bounding box from topmost leaf to bottommost leaf
    y_min_leaf = y_dict[leaves_sorted[0]]
    y_max_leaf = y_dict[leaves_sorted[-1]]

    # small pad
    global_pad = 0.2*(y_max_leaf - y_min_leaf)
    y_min = y_min_leaf - global_pad
    y_max = y_max_leaf + global_pad

    total_height = y_max - y_min
    chunk = total_height / N

    # cluster labels
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels)<N:
        cluster_labels += [cluster_labels[-1]]*(N-len(cluster_labels))

    # define tri_x line
    from math import inf
    if side=="right":
        x_leaf_max = max(x_dict[l] for l in leaves)
        tri_x = x_leaf_max + triangle_offset
        # place label box further out
        label_box_dx = +5 # shift label box to the *right* of tri_x
    else:
        x_leaf_min = min(x_dict[l] for l in leaves)
        tri_x = x_leaf_min - triangle_offset
        # shift label box slightly more to the left
        label_box_dx = -10

    # measure axis data-lims for label box size
    xlim = ax.get_xlim()
    data_w = xlim[1]-xlim[0]
    box_aspect = 2.0
    box_w_fraction = 0.08
    box_w = box_w_fraction*data_w

    from matplotlib.patches import Polygon, Rectangle
    for i, leaf in enumerate(leaves_sorted):
        top = y_min + i*chunk
        bottom = top + chunk

        lx = x_dict[leaf]
        ly = y_dict[leaf]

        # define triangle
        if side=="right":
            tri_verts = [(lx, ly),(tri_x, top),(tri_x, bottom)]
        else:
            tri_verts = [(lx, ly),(tri_x, top),(tri_x, bottom)]

        poly = Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5
        )
        ax.add_patch(poly)

        # place label box entirely outside the triangle
        midy = 0.5*(top+bottom)
        box_h = 0.6*chunk
        box_y = midy - 0.5*box_h

        if side=="right":
            # label box to the right of tri_x
            box_x = tri_x + label_box_dx
        else:
            # label box to the left of tri_x minus box_w
            box_x = tri_x + 5 + label_box_dx - box_w

        rect = Rectangle(
            (box_x, box_y),
            box_w, box_h,
            facecolor=triangle_color,
            alpha=0.3,
            edgecolor="none",
            zorder=6
        )
        ax.add_patch(rect)

        ax.text(
            box_x + 0.5*box_w,
            box_y + 0.5*box_h,
            cluster_labels[i],
            ha="center", va="center",
            fontsize=label_fontsize, color="black",
            zorder=7
        )

    # optionally re-adjust y-lims so we see entire top/bottom
    ax.set_ylim(y_min, y_max)

#################################
# 4) SINGLE SCALE BAR WITH SCALING
#################################

def pick_scale_length(xrange: float) -> float:
    """
    Return a 'nice' scale bar ~20% of xrange, e.g. 0.1, 0.2, 0.5, 1, 2, 5, 10, ...
    """
    if xrange <= 0:
        return 0.1
    target = 0.2*xrange
    p = 10**(np.floor(np.log10(target)))
    factor = target/p
    if factor<2:
        return p
    elif factor<5:
        return 2*p
    else:
        return 5*p

def add_scale_bar(
    ax: plt.Axes,
    x_dicts: List[dict],
    scale_factors: List[float],
    location="bottomleft",
    line_width=2.0,
    color="black",
    fontsize=10,
):
    """
    Combine x-ranges from x_dicts in original scale. Then pick a scale bar ~20%.
    BUT we must multiply that bar length by the scale factor used for data coords.

    x_dicts: the x-dicts after scaling
    scale_factors: each x-dict's scale factor
    We assume all subplots used the same scale factor or we pick the first.

    By default, if you are using the same scale_factor for both trees, we can
    do scale_factor = scale_factors[0] and ignore the second.
    """
    if not scale_factors:
        return
    # assume the same factor for both if user wants
    scale_factor = scale_factors[0]

    # We want the original min_x, max_x *before* scaling. So let's do:
    # we can do min_x = min( x_dict[c]/scale_factor ) etc. if we didn't store original
    # Or store them if needed. For simplicity, let's just do what we can:
    min_x = float("inf")
    max_x = float("-inf")
    for i, xd in enumerate(x_dicts):
        fac = scale_factors[i]
        # we revert the scaling by dividing by fac to get the original coords
        xs_orig = [x/fac for x in xd.values()]
        mi, ma = min(xs_orig), max(xs_orig)
        if mi<min_x: min_x=mi
        if ma>max_x: max_x=ma

    data_range_original = max_x - min_x
    bar_len_orig = pick_scale_length(data_range_original)
    # the bar in data coords after scaling is bar_len_orig * scale_factor
    bar_len_data = bar_len_orig * scale_factor

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_range = xlim[1]-xlim[0]
    y_range = ylim[1]-ylim[0] if ylim[1]>ylim[0] else (ylim[0]-ylim[1])
    xmarg = 0.02*x_range
    ymarg = 0.02*y_range

    if "bottom" in location:
        if ax.yaxis_inverted():
            y0 = ylim[0]+ymarg
        else:
            y0 = ylim[1]-ymarg
    else:
        # top
        if ax.yaxis_inverted():
            y0 = ylim[1]-ymarg
        else:
            y0 = ylim[0]+ymarg

    if "left" in location:
        x0 = xlim[0]+xmarg
    else:
        x0 = xlim[1]-xmarg - bar_len_data

    x1 = x0+bar_len_data
    ax.plot([x0,x1],[y0,y0], lw=line_width, color=color, zorder=10)
    # label the original bar length
    ax.text(
        0.5*(x0+x1), y0,
        f"{bar_len_orig:g}",
        ha="center",
        va="bottom" if not ax.yaxis_inverted() else "top",
        fontsize=fontsize,
        color=color,
        zorder=11
    )


def add_uniform_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_offset: float = 1.0,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    cluster_labels: Optional[List[str]] = None,
    label_fontsize: int = 10,
):
    """
    - Compute bounding box from leaves only => uniform chunk in vertical dimension.
    - Triangles each have the same vertical height.
    - Place label boxes *outside* the triangle (to the extreme right or left),
      and scale the label box width according to the text length so that
      longer labels get bigger boxes.

    The top leaf gets the first chunk, the bottom leaf gets the last chunk,
    so they exactly fill the leaf range with optional padding.

    We'll approximate label box width as: W ~ (#chars in label) * a fraction of data_w.
    """

    # 1) Gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)

    # 2) bounding box from topmost leaf to bottommost leaf
    y_min_leaf = y_dict[leaves_sorted[0]]
    y_max_leaf = y_dict[leaves_sorted[-1]]

    # 3) small pad
    global_pad = 0.2 * (y_max_leaf - y_min_leaf)
    y_min = y_min_leaf - global_pad
    y_max = y_max_leaf + global_pad

    total_height = y_max - y_min
    chunk = total_height / N

    # 4) cluster labels
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels) < N:
        cluster_labels += [cluster_labels[-1]] * (N - len(cluster_labels))

    # 5) define tri_x line
    if side == "right":
        x_leaf_max = max(x_dict[l] for l in leaves)
        tri_x = x_leaf_max + triangle_offset
        label_box_dx = +5  # shift label box to the right of tri_x
    else:
        x_leaf_min = min(x_dict[l] for l in leaves)
        tri_x = x_leaf_min - triangle_offset
        label_box_dx = -5  # shift label box to the left of tri_x

    # 6) measure axis data-lims for approximate sizing
    xlim = ax.get_xlim()
    data_w = xlim[1] - xlim[0]

    from matplotlib.patches import Polygon, Rectangle

    for i, leaf in enumerate(leaves_sorted):
        top = y_min + i * chunk
        bottom = top + chunk

        lx = x_dict[leaf]
        ly = y_dict[leaf]

        # Triangles: apex (leaf_x, leaf_y) to (tri_x, top) + (tri_x, bottom)
        if side == "right":
            tri_verts = [(lx, ly), (tri_x, top), (tri_x, bottom)]
        else:
            tri_verts = [(lx, ly), (tri_x, top), (tri_x, bottom)]

        poly = Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5,
        )
        ax.add_patch(poly)

        # 7) place label box entirely outside the triangle
        midy = 0.5 * (top + bottom)
        box_h = 0.6 * chunk
        box_y = midy - 0.5 * box_h

        label_str = cluster_labels[i]
        # approximate the # of chars
        n_chars = len(label_str)
        # define how many data units per char (tweak as desired)
        # e.g. 0.008 * data_w => each char is ~0.8% of total axis width
        # the bigger this factor, the bigger the box for long labels
        data_units_per_char = 0.008 * data_w

        box_w = max(20.0, n_chars * data_units_per_char)
        # 'max(20.0, ... )' ensures a minimum absolute width in data coords

        if side == "right":
            box_x = tri_x + label_box_dx
        else:
            box_x = tri_x + label_box_dx - box_w

        # rectangle
        rect = Rectangle(
            (box_x, box_y),
            box_w,
            box_h,
            facecolor=triangle_color,
            alpha=0.3,
            edgecolor="none",
            zorder=6,
        )
        ax.add_patch(rect)

        # text inside
        ax.text(
            box_x + 0.5 * box_w,
            box_y + 0.5 * box_h,
            label_str,
            ha="center",
            va="center",
            fontsize=label_fontsize,
            color="black",
            zorder=7,
        )

    # 8) optionally re-adjust y-lims so we see entire top/bottom
    ax.set_ylim(y_min, y_max)


def add_uniform_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_offset: float = 1.0,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    cluster_labels: Optional[List[str]] = None,
    label_fontsize: int = 10,
):
    """
    Each leaf gets the same vertical chunk from top->bottom leaf range.
    Triangles are drawn from leaf apex => (tri_x, top) and (tri_x, bottom).
    The cluster label is placed outside the triangle using 'bbox' in ax.text()
    so that a background box is automatically sized around the text.

    'triangle_offset': how far horizontally from the furthest leaf x we place tri_x.
    This can be large if the tree is shrunk, so the triangles appear longer.
    """
    # Gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)

    # bounding box from top leaf to bottom leaf
    y_min_leaf = y_dict[leaves_sorted[0]]
    y_max_leaf = y_dict[leaves_sorted[-1]]

    # small pad
    global_pad = 0.2 * (y_max_leaf - y_min_leaf)
    y_min = y_min_leaf - global_pad
    y_max = y_max_leaf + global_pad

    total_height = y_max - y_min
    chunk = total_height / N

    # cluster labels
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels) < N:
        cluster_labels += [cluster_labels[-1]] * (N - len(cluster_labels))

    # define tri_x
    leaves_x_positions = [x_dict[l] for l in leaves]
    if side == "right":
        x_leaf_max = max(leaves_x_positions)
        tri_x = x_leaf_max + triangle_offset
        label_dx = 5.0  # shift label text further to the right
    else:
        x_leaf_min = min(leaves_x_positions)
        tri_x = x_leaf_min - triangle_offset
        label_dx = -5.0  # shift label text further to the left

    from matplotlib.patches import Polygon

    for i, leaf in enumerate(leaves_sorted):
        top = y_min + i * chunk
        bottom = top + chunk
        lx = x_dict[leaf]
        ly = y_dict[leaf]

        # define triangle
        if side == "right":
            tri_verts = [(lx, ly), (tri_x, top), (tri_x, bottom)]
        else:
            tri_verts = [(lx, ly), (tri_x, top), (tri_x, bottom)]

        poly = Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5,
        )
        ax.add_patch(poly)

        # midpoint in that chunk
        mid_y = 0.5 * (top + bottom)

        # place text outside the triangle with a bounding box
        if side == "right":
            label_x = tri_x + label_dx
        else:
            label_x = tri_x + label_dx

        label_str = cluster_labels[i]

        # we let matplotlib auto-size the bounding box for the text:
        ax.text(
            label_x,
            mid_y,
            label_str,
            ha="left" if side == "right" else "right",  # align away from triangle
            va="center",
            fontsize=label_fontsize,
            color="black",
            zorder=7,
            bbox=dict(
                boxstyle="square,pad=0.4",  # or "square"
                fc=triangle_color,
                alpha=0.3,
                ec="none",
            ),
        )

    # re-adjust y-lims
    ax.set_ylim(y_min, y_max)


#################################
# 5) DEMO
#################################

def demo():
    import Bio.Phylo as bp
    # Example newick with small branch lengths
    newick_str = "((A:1,B:2):3,(C:0.5,D:0.8):1.2);"
    tree_left = bp.read(StringIO(newick_str), "newick")
    tree_right = bp.read(StringIO(newick_str), "newick")

    fig, axes = plt.subplots(1,2, figsize=(16,6))

    # 1) Plot left tree with scale_factor=50 (for example)
    xdict_left, ydict_left, scale_factor_left = plot_tree(
        tree_left,
        ax=axes[0],
        flip_x=False,
        scale_factor=50.0,  # scale up small branch lengths by 50
        line_color="black",
        line_width=2.0
    )
    all_left = list(tree_left.find_clades())

    # 2) Add uniform triangles on the right side, labels outside
    add_uniform_triangles(
        ax=axes[0],
        x_dict=xdict_left,
        y_dict=ydict_left,
        all_clades=all_left,
        side="right",
        triangle_offset=20.0,  # bigger offset in data coords
        triangle_color="green",
        triangle_alpha=0.8,
        # cluster_labels=["C1","C2","C3","C4"],
        label_fontsize=10
    )
    axes[0].set_title("Left: Normal Tree, scale_factor=50, Triangles on Right", fontsize=11)

    # 3) Plot right tree with flip_x=True, same scale factor
    xdict_right, ydict_right, scale_factor_right = plot_tree(
        tree_right,
        ax=axes[1],
        flip_x=True,
        scale_factor=50.0,  # same scale up
        line_color="black",
        line_width=2.0
    )
    all_right = list(tree_right.find_clades())

    add_uniform_triangles(
        ax=axes[1],
        x_dict=xdict_right,
        y_dict=ydict_right,
        all_clades=all_right,
        side="left",
        triangle_offset=20.0,
        triangle_color="orange",
        triangle_alpha=0.8,
        cluster_labels=["C1","C2","C3","C466666"],
        label_fontsize=10
    )
    axes[1].set_title("Right: Flipped Tree, scale_factor=50, Triangles on Left", fontsize=11)

    # 4) Single scale bar in the left subplot
    # We'll pass the x_dicts plus their scale_factors so we can revert to original coords
    add_scale_bar(
        ax=axes[0],
        x_dicts=[xdict_left, xdict_right],
        scale_factors=[scale_factor_left, scale_factor_right],
        location="bottomleft",
        line_width=2.0,
        color="black",
        fontsize=9
    )

    plt.tight_layout()
    plt.show()

if __name__=="__main__":
    demo()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from Bio import Phylo
from matplotlib.patches import Polygon, Rectangle
from io import StringIO
from typing import List, Optional

#################################
# 1) UTILS: get_x_positions, get_y_positions, flip_x_positions
#################################

def get_x_positions(tree):
    """
    Return a dict mapping each clade to its x-position (distance from root),
    forcing the root's branch_length=0 to avoid extra stubs.
    """
    # Force root branch length = 0
    if tree.root.branch_length:
        tree.root.branch_length = 0.0

    x_dict = {}
    def assign_x(clade, curx):
        x_dict[clade] = curx
        for child in clade.clades:
            bl = child.branch_length or 0.0
            assign_x(child, curx + bl)
    assign_x(tree.root, 0.0)
    return x_dict


def get_y_positions(tree):
    """
    Return a dict mapping each clade to a y-position in top-to-bottom order for leaves.
    Internal nodes go at the mean of their children.
    """
    y_dict = {}
    y_counter = 0
    def assign_y(clade):
        nonlocal y_counter
        for child in clade.clades:
            assign_y(child)
        if not clade.clades:  # leaf
            y_dict[clade] = y_counter
            y_counter += 1
        else:
            child_ys = [y_dict[ch] for ch in clade.clades]
            y_dict[clade] = np.mean(child_ys)
    assign_y(tree.root)
    return y_dict


def flip_x_positions(x_dict):
    """
    Reflect the x-values around their midpoint so the tree is horizontally reversed.
    """
    xs = list(x_dict.values())
    xmin, xmax = min(xs), max(xs)
    mid = 0.5*(xmin + xmax)
    for c in x_dict:
        old = x_dict[c]
        x_dict[c] = mid - (old - mid)

def scale_x_positions(x_dict, factor: float):
    """
    Multiply all x-coordinates by 'factor' to scale up small branch lengths visually.
    This does not change the 'units' logically, but we must handle the scale bar carefully.
    """
    for c in x_dict:
        x_dict[c] *= factor

#################################
# 2) PLOTTING THE TREE LINES ONLY
#################################

def plot_tree(
    tree,
    ax: plt.Axes,
    flip_x: bool=False,
    scale_factor: float=1.0,
    line_color: str="black",
    line_width: float=1.0
):
    """
    Draws a cladogram (horizontal lines + vertical lines for branches),
    no leaf markers or names. If flip_x=True, data-coords are mirrored horizontally.
    Also multiply all x positions by 'scale_factor'.
    Returns (x_dict, y_dict, scale_factor).
    """
    # 1) get original x,y
    x_dict = get_x_positions(tree)
    y_dict = get_y_positions(tree)

    # 2) optionally flip
    if flip_x:
        flip_x_positions(x_dict)
    # 3) optionally scale up
    if scale_factor != 1.0:
        scale_x_positions(x_dict, scale_factor)

    # gather line segments
    h_segments = []
    v_segments = []

    def add_hline(x1, x2, y):
        h_segments.append(((x1,y),(x2,y)))
    def add_vline(x, y1, y2):
        # skip near-zero lines
        if abs(y2 - y1) < 1e-9:
            return
        v_segments.append(((x,y1),(x,y2)))

    def draw_clade(clade, xstart, is_root=False):
        xh = x_dict[clade]
        yh = y_dict[clade]
        if not is_root:
            add_hline(xstart, xh, yh)
        if clade.clades:
            y_bot = y_dict[clade.clades[-1]]
            y_top = y_dict[clade.clades[0]]
            v_segments.append(((xh,y_bot),(xh,y_top)))
            for child in clade.clades:
                draw_clade(child, xh)

    draw_clade(tree.root, 0.0, is_root=True)

    import matplotlib.collections as mc
    hc = mc.LineCollection(h_segments, colors=line_color, linewidths=line_width)
    vc = mc.LineCollection(v_segments, colors=line_color, linewidths=line_width)
    ax.add_collection(hc)
    ax.add_collection(vc)

    # define lims
    all_x = list(x_dict.values())
    all_y = list(y_dict.values())
    xmin, xmax = min(all_x), max(all_x)
    ymin, ymax = min(all_y), max(all_y)

    xmargin = 0.1*(xmax - xmin) if (xmax>xmin) else 1
    ymargin = 0.5
    ax.set_xlim(xmin - xmargin, xmax + xmargin)
    ax.set_ylim(ymax + ymargin, ymin - ymargin)

    # remove ticks/spines
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    return x_dict, y_dict, scale_factor

#################################
# 3) UNIFORM TRIANGLES + LABELS OUTSIDE
#################################
def add_identical_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_width: float = 10.0,
    triangle_height: float = 6.0,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    cluster_labels: Optional[List[str]] = None,
    label_fontsize: int = 10,
):
    """
    Draw the *same* sized triangle for each leaf, ignoring any uniform vertical tiling.
    Each leaf's triangle has a base = 'triangle_width' in data coords,
    and a total height = 'triangle_height' in data coords.

    We also place a label box next to each triangle; *all* label boxes have the
    SAME size, determined by the longest label among cluster_labels.

    Args:
      side: "right" or "left" => if "right", apex is at (leaf_x, leaf_y) and
            the base extends to the right in data coords.
      triangle_width, triangle_height: how big each triangle is, in data coords.
      cluster_labels: text for each leaf. If fewer than leaves, repeat last.
      label_fontsize: text size for labels.

    Example usage:
      add_identical_triangles(ax, x_dict, y_dict, all_clades,
                              triangle_width=15, triangle_height=8, side="right")
    """

    # 1) Gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return  # no leaves => nothing to do

    # 2) Sort leaves by y if you want them in ascending order (not strictly needed)
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)

    # 3) If cluster_labels not specified, create defaults
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    # If fewer labels than leaves, repeat the last label
    if len(cluster_labels) < N:
        cluster_labels += [cluster_labels[-1]]*(N-len(cluster_labels))

    # 4) Find the LONGEST label => define a single bounding-box size for ALL
    max_label_len = max(len(lbl) for lbl in cluster_labels)
    # You can guess a box width from # of chars:
    # e.g. each char => 0.008 * axis width. Or do a simpler approach:
    xlim = ax.get_xlim()
    data_w = xlim[1] - xlim[0]

    # Let's define a single bounding box width/height:
    # e.g. a  minimum 20 data units wide or #chars * factor
    label_char_factor = 0.01 * data_w
    box_w = max(20.0, label_char_factor * max_label_len)
    box_h = 1.2 * label_fontsize  # e.g. each char row ~1.2 * font size in data coords

    # If you want all label boxes the same size for all leaves
    # we don't recompute inside the loop

    import matplotlib.patches as mp

    # 5) For each leaf, draw a *fixed-size* triangle
    for i, leaf in enumerate(leaves_sorted):
        leaf_x = x_dict[leaf]
        leaf_y = y_dict[leaf]

        # A) Define the triangle. We choose apex at (leaf_x, leaf_y).
        # If side="right", the base extends triangle_width to the right.
        # The triangle center's vertical position is leaf_y, so half the total
        # height extends above and half extends below.
        half_h = 0.5 * triangle_height

        if side=="right":
            # apex is left, base is right
            tri_verts = [
                (leaf_x, leaf_y),  # apex
                (leaf_x + triangle_width, leaf_y + half_h),
                (leaf_x + triangle_width, leaf_y - half_h),
            ]
            label_x = leaf_x + triangle_width + 5  # small offset from the triangle base
            ha = "left"
        else:
            # side="left" => apex is right, base extends left
            tri_verts = [
                (leaf_x, leaf_y),  # apex
                (leaf_x - triangle_width, leaf_y + half_h),
                (leaf_x - triangle_width, leaf_y - half_h),
            ]
            label_x = leaf_x - triangle_width - 5  # offset from apex
            ha = "right"

        # B) Draw the triangle
        poly = mp.Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5,
        )
        ax.add_patch(poly)

        # C) Place the label box => same size for all leaves
        label_str = cluster_labels[i]
        label_y = leaf_y  # center the label at leaf_y
        # We'll do a rectangle or use text's bbox? Let's do a single bounding box for all:
        # We'll simply do a text with a manual rectangle or do a "bbox" param.
        # Easiest: use text's bbox so it auto-sizes? But you want a FIXED size for all?
        # => We'll manually place a rectangle, then text in it:

        box_y = label_y - 0.5*box_h

        rect = mp.Rectangle(
            (label_x, box_y),
            box_w, box_h,
            facecolor=triangle_color,
            alpha=0.3,
            edgecolor="none",
            zorder=6
        )
        ax.add_patch(rect)

        # mid_y = 0.5 * (top + bottom)

        # # place text outside the triangle with a bounding box
        # if side == "right":
        #     label_x = tri_x + label_dx
        # else:
        #     label_x = tri_x + label_dx

        label_str = cluster_labels[i]

        # we let matplotlib auto-size the bounding box for the text:
        ax.text(
            x, y,
            label_str,
            fontsize=label_fontsize,
            va="center",
            ha="center",
            bbox=dict(
                boxstyle="round,pad=0.3",  # or "square,pad=0.3"
                fc="red",      # face color
                alpha=0.3,
                ec="none"      # no edge
            )
        )


    # Optionally re-adjust y-lims if you want some extra margin
    all_yvals = [y_dict[leaf] for leaf in leaves]
    min_y, max_y = min(all_yvals), max(all_yvals)
    margin = 0.2*(max_y - min_y) if (max_y>min_y) else 1
    ax.set_ylim(max_y + margin, min_y - margin)


#################################
# 4) SINGLE SCALE BAR WITH SCALING
#################################

def pick_scale_length(xrange: float) -> float:
    """
    Return a 'nice' scale bar ~20% of xrange, e.g. 0.1, 0.2, 0.5, 1, 2, 5, 10, ...
    """
    if xrange <= 0:
        return 0.1
    target = 0.2*xrange
    p = 10**(np.floor(np.log10(target)))
    factor = target/p
    if factor<2:
        return p
    elif factor<5:
        return 2*p
    else:
        return 5*p

def add_scale_bar(
    ax: plt.Axes,
    x_dicts: List[dict],
    scale_factors: List[float],
    location="bottomleft",
    line_width=2.0,
    color="black",
    fontsize=10,
):
    """
    Combine x-ranges from x_dicts in original scale. Then pick a scale bar ~20%.
    BUT we must multiply that bar length by the scale factor used for data coords.

    x_dicts: the x-dicts after scaling
    scale_factors: each x-dict's scale factor
    We assume all subplots used the same scale factor or we pick the first.

    By default, if you are using the same scale_factor for both trees, we can
    do scale_factor = scale_factors[0] and ignore the second.
    """
    if not scale_factors:
        return
    # assume the same factor for both if user wants
    scale_factor = scale_factors[0]

    # We want the original min_x, max_x *before* scaling. So let's do:
    # we can do min_x = min( x_dict[c]/scale_factor ) etc. if we didn't store original
    # Or store them if needed. For simplicity, let's just do what we can:
    min_x = float("inf")
    max_x = float("-inf")
    for i, xd in enumerate(x_dicts):
        fac = scale_factors[i]
        # we revert the scaling by dividing by fac to get the original coords
        xs_orig = [x/fac for x in xd.values()]
        mi, ma = min(xs_orig), max(xs_orig)
        if mi<min_x: min_x=mi
        if ma>max_x: max_x=ma

    data_range_original = max_x - min_x
    bar_len_orig = pick_scale_length(data_range_original)
    # the bar in data coords after scaling is bar_len_orig * scale_factor
    bar_len_data = bar_len_orig * scale_factor

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_range = xlim[1]-xlim[0]
    y_range = ylim[1]-ylim[0] if ylim[1]>ylim[0] else (ylim[0]-ylim[1])
    xmarg = 0.02*x_range
    ymarg = 0.02*y_range

    if "bottom" in location:
        if ax.yaxis_inverted():
            y0 = ylim[0]+ymarg
        else:
            y0 = ylim[1]-ymarg
    else:
        # top
        if ax.yaxis_inverted():
            y0 = ylim[1]-ymarg
        else:
            y0 = ylim[0]+ymarg

    if "left" in location:
        x0 = xlim[0]+xmarg
    else:
        x0 = xlim[1]-xmarg - bar_len_data

    x1 = x0+bar_len_data
    ax.plot([x0,x1],[y0,y0], lw=line_width, color=color, zorder=10)
    # label the original bar length
    ax.text(
        0.5*(x0+x1), y0,
        f"{bar_len_orig:g}",
        ha="center",
        va="bottom" if not ax.yaxis_inverted() else "top",
        fontsize=fontsize,
        color=color,
        zorder=11
    )
    if "bottom" in location:
        if ax.yaxis_inverted():
            ax.set_ylim(ylim[0], ylim[1] + 2 * ymarg)
        else:
            ax.set_ylim(ylim[0] - 2 * ymarg, ylim[1])
    else:
        if ax.yaxis_inverted():
            ax.set_ylim(ylim[0] - 2 * ymarg, ylim[1])
        else:
            ax.set_ylim(ylim[0], ylim[1] + 2 * ymarg)

    if "left" in location:
        ax.set_xlim(xlim[0] - 2 * xmarg, xlim[1])
    else:
        ax.set_xlim(xlim[0], xlim[1] + 2 * xmarg)


def add_uniform_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_offset: float = 1.0,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    cluster_labels: Optional[List[str]] = None,
    label_fontsize: int = 10,
):
    """
    - Compute bounding box from leaves only => uniform chunk in vertical dimension.
    - Triangles each have the same vertical height.
    - Place label boxes *outside* the triangle (to the extreme right or left),
      and scale the label box width according to the text length so that
      longer labels get bigger boxes.

    The top leaf gets the first chunk, the bottom leaf gets the last chunk,
    so they exactly fill the leaf range with optional padding.

    We'll approximate label box width as: W ~ (#chars in label) * a fraction of data_w.
    """

    # 1) Gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)

    # 2) bounding box from topmost leaf to bottommost leaf
    y_min_leaf = y_dict[leaves_sorted[0]]
    y_max_leaf = y_dict[leaves_sorted[-1]]

    # 3) small pad
    global_pad = 0.2 * (y_max_leaf - y_min_leaf)
    y_min = y_min_leaf - global_pad
    y_max = y_max_leaf + global_pad

    total_height = y_max - y_min
    chunk = total_height / N

    # 4) cluster labels
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels) < N:
        cluster_labels += [cluster_labels[-1]] * (N - len(cluster_labels))

    # 5) define tri_x line
    if side == "right":
        x_leaf_max = max(x_dict[l] for l in leaves)
        tri_x = x_leaf_max + triangle_offset
        label_box_dx = +5  # shift label box to the right of tri_x
    else:
        x_leaf_min = min(x_dict[l] for l in leaves)
        tri_x = x_leaf_min - triangle_offset
        label_box_dx = -5  # shift label box to the left of tri_x

    # 6) measure axis data-lims for approximate sizing
    xlim = ax.get_xlim()
    data_w = xlim[1] - xlim[0]

    from matplotlib.patches import Polygon, Rectangle

    for i, leaf in enumerate(leaves_sorted):
        top = y_min + i * chunk
        bottom = top + chunk

        lx = x_dict[leaf]
        ly = y_dict[leaf]

        # Triangles: apex (leaf_x, leaf_y) to (tri_x, top) + (tri_x, bottom)
        if side == "right":
            tri_verts = [(lx, ly), (tri_x, top), (tri_x, bottom)]
        else:
            tri_verts = [(lx, ly), (tri_x, top), (tri_x, bottom)]

        poly = Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5,
        )
        ax.add_patch(poly)

        # 7) place label box entirely outside the triangle
        midy = 0.5 * (top + bottom)
        box_h = 0.6 * chunk
        box_y = midy - 0.5 * box_h

        label_str = cluster_labels[i]
        # approximate the # of chars
        n_chars = len(label_str)
        # define how many data units per char (tweak as desired)
        # e.g. 0.008 * data_w => each char is ~0.8% of total axis width
        # the bigger this factor, the bigger the box for long labels
        data_units_per_char = 0.008 * data_w

        box_w = max(20.0, n_chars * data_units_per_char)
        # 'max(20.0, ... )' ensures a minimum absolute width in data coords

        if side == "right":
            box_x = tri_x + label_box_dx
        else:
            box_x = tri_x + label_box_dx - box_w

        # rectangle
        rect = Rectangle(
            (box_x, box_y),
            box_w,
            box_h,
            facecolor=triangle_color,
            alpha=0.3,
            edgecolor="none",
            zorder=6,
        )
        ax.add_patch(rect)

        mid_y = 0.5 * (top + bottom)

        # place text outside the triangle with a bounding box
        if side == "right":
            label_x = tri_x + label_dx
        else:
            label_x = tri_x + label_dx

        label_str = cluster_labels[i]

        # we let matplotlib auto-size the bounding box for the text:
        ax.text(
            label_x,
            mid_y,
            label_str,
            ha="left" if side == "right" else "right",  # align away from triangle
            va="center",
            fontsize=label_fontsize,
            color="black",
            zorder=7,
            bbox=dict(
                boxstyle="square,pad=0.4",  # or "square"
                fc=triangle_color,
                alpha=0.3,
                ec="none",
            ),
        )

    # re-adjust y-lims
    ax.set_ylim(y_min, y_max)


#################################
# 5) DEMO
#################################
def add_identical_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_width: float = 10.0,
    triangle_height: float = 0.5,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    cluster_labels: Optional[List[str]] = None,
    label_fontsize: int = 10,
):
    """
    Draw the *same* sized triangle for each leaf, ignoring uniform vertical tiling.
    Each leaf's triangle has a base = 'triangle_width' in data coords,
    and a total height = 'triangle_height' in data coords.

    Then place a label next to each triangle using ax.text(..., bbox=...),
    which automatically sizes the box to the text (Approach A).
    So each label's bounding box is sized specifically to that label (not uniform).

    Args:
      side: "right" or "left"
        - If "right", apex is at (leaf_x, leaf_y) and the base extends horizontally
          'triangle_width' to the right in data coords.
        - If "left", apex is at (leaf_x, leaf_y) and extends to the left.
      triangle_width, triangle_height: how big each triangle is, in data coords.
      cluster_labels: text for each leaf. If fewer than leaves, repeat last.
      label_fontsize: text size for labels.
    """

    # 1) gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return  # no leaves => nothing to do

    # 2) optionally sort leaves by ascending y
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)

    # 3) define cluster_labels if not given
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels) < N:
        cluster_labels += [cluster_labels[-1]] * (N - len(cluster_labels))

    import matplotlib.patches as mp

    half_h = 0.5 * triangle_height  # half the triangle's total height

    # 4) For each leaf, draw an identical triangle
    for i, leaf in enumerate(leaves_sorted):
        leaf_x = x_dict[leaf]
        leaf_y = y_dict[leaf]

        if side == "right":
            # apex at (leaf_x, leaf_y), base extends triangle_width to the right
            tri_verts = [
                (leaf_x, leaf_y),
                (leaf_x + triangle_width, leaf_y + half_h),
                (leaf_x + triangle_width, leaf_y - half_h),
            ]
            # place label to the right of the base
            label_x = leaf_x + triangle_width + 5.0  # small horizontal offset
            label_ha = "left"
        else:
            # apex at (leaf_x, leaf_y), base extends triangle_width to the left
            tri_verts = [
                (leaf_x, leaf_y),
                (leaf_x - triangle_width, leaf_y + half_h),
                (leaf_x - triangle_width, leaf_y - half_h),
            ]
            label_x = leaf_x - triangle_width - 5.0
            label_ha = "right"

        # draw the triangle
        poly = mp.Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5,
        )
        ax.add_patch(poly)

        # 5) place the label with an auto‐sized bounding box
        label_str = cluster_labels[i]
        ax.text(
            label_x,  # horizontally outside the triangle
            leaf_y,  # vertically at the same y as the apex
            label_str,
            fontsize=label_fontsize,
            color="black",
            va="center",
            ha=label_ha,
            bbox=dict(
                boxstyle="square,pad=0.4",  # or "square,pad=0.2", etc.
                fc=triangle_color,  # same color as triangle
                alpha=0.3,
                ec="none",
            ),
            zorder=6,
        )

    # optionally define axis y-limits so we can see everything
    # e.g. expand top & bottom a bit
    all_yvals = [y_dict[leaf] for leaf in leaves]
    if all_yvals:
        min_y, max_y = min(all_yvals), max(all_yvals)
        pad = 0.2 * (max_y - min_y) if max_y > min_y else 1.0
        ax.set_ylim(max_y + pad, min_y - pad)
    # optionally also x-limits, etc.

def demo():
    import Bio.Phylo as bp
    # Example newick with small branch lengths
    newick_str = "((A:1,B:2):3,(C:0.5,D:0.8):1.2);"
    tree_left = bp.read(StringIO(newick_str), "newick")
    tree_right = bp.read(StringIO(newick_str), "newick")

    fig, axes = plt.subplots(1,2, figsize=(16,6))

    # 1) Plot left tree with scale_factor=50 (for example)
    xdict_left, ydict_left, scale_factor_left = plot_tree(
        tree_left,
        ax=axes[0],
        flip_x=False,
        scale_factor=50.0,  # scale up small branch lengths by 50
        line_color="black",
        line_width=2.0
    )
    all_left = list(tree_left.find_clades())

    # 2) Add uniform triangles on the right side, labels outside
    add_identical_triangles(
        ax=axes[0],
        x_dict=xdict_left,
        y_dict=ydict_left,
        all_clades=all_left,
        side="right",
        triangle_width= 15.0,
        triangle_height= 0.5,
        triangle_color="green",
        triangle_alpha=0.8,
        # cluster_labels=["C1","C2","C3","C4"],
        label_fontsize=10
    )
    axes[0].set_title("Left: Normal Tree, scale_factor=50, Triangles on Right", fontsize=11)

    # 3) Plot right tree with flip_x=True, same scale factor
    xdict_right, ydict_right, scale_factor_right = plot_tree(
        tree_right,
        ax=axes[1],
        flip_x=True,
        scale_factor=50.0,  # same scale up
        line_color="black",
        line_width=2.0
    )
    all_right = list(tree_right.find_clades())

    add_identical_triangles(
        ax=axes[1],
        x_dict=xdict_right,
        y_dict=ydict_right,
        all_clades=all_right,
        side="left",
        # triangle_offset=20.0,
        # triangle_color="orange",
        # triangle_alpha=0.8,
        cluster_labels=["C1","C2","C3","C466666"],
        label_fontsize=10
    )
    axes[1].set_title("Right: Flipped Tree, scale_factor=50, Triangles on Left", fontsize=11)

    # 4) Single scale bar in the left subplot
    # We'll pass the x_dicts plus their scale_factors so we can revert to original coords
    add_scale_bar(
        ax=axes[0],
        x_dicts=[xdict_left, xdict_right],
        scale_factors=[scale_factor_left, scale_factor_right],
        location="bottomleft",
        line_width=5.0,
        color="black",
        fontsize=10
    )

    plt.tight_layout()
    plt.show()


if __name__=="__main__":
    demo()